# Chapter 19 — Deep Feedforward Neural Networks I: Optimization Methods

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part3_neural_networks/ch19_deep_feedforward_neural_networks_i/ch19_notebook.ipynb)

Companion notebook for **Chapter 19** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Plain SGD is slow on elongated error surfaces because it takes equally small steps in every direction.
- Momentum accumulates velocity in consistently downhill directions, so it accelerates along the valley floor.
- AdaGrad accumulates all past squared gradients, so its learning rate eventually decays to almost nothing.
- RMSProp fixes this with a moving average that forgets old gradients; Adam combines momentum with RMSProp.
- L1 regularization drives weights to exactly zero (giving feature selection); L2 shrinks them smoothly without zeroing.

## Example 1: SGD vs Momentum vs Adam — Optimizer Convergence

We minimize a 2-D quadratic loss E(w) = w1² + 50·w2² starting at w = [5, 5]. The '50' on w2 makes a narrow ravine: steep across w2, almost flat along w1. Plain SGD uses one step size for both weights, and the steep direction sets the limit. The first row shows that a step of 0.021 makes w2 overshoot further every time and blow up. With a safe step of 0.009, SGD crawls along the flat w1 direction and is still at loss 0.66 after 100 steps. Momentum keeps the same step size but builds up speed along w1 and finishes far ahead (0.0009). Adam divides each weight's step by the size of its recent gradients, so the steep w2 gets a smaller effective step and the flat w1 a larger one, and both move at a similar pace. It beats SGD clearly (0.024), but on this small, smooth bowl a well-tuned momentum wins. Adam's real advantage is that it works well across many problems with little tuning, which is why it is the usual default.

In [1]:
import numpy as np

print("=== Optimizer Comparison: Minimizing E(w) = w1^2 + 50*w2^2 ===\n")
print("Starting at w=[5.0, 5.0], optimal = [0.0, 0.0]")
print("A narrow ravine: steep in w2 (gradient 100*w2), almost flat in w1 (gradient 2*w1)\n")

loss = lambda w: w[0]**2 + 50*w[1]**2
grad = lambda w: np.array([2*w[0], 100*w[1]])

# SGD: the steep w2 direction caps the step size (above eta = 0.02 it blows up)
def make_sgd(eta=0.009):
    def update(w, g): return w - eta * g
    return update

# Momentum: same step size, but velocity builds up along the flat w1 direction
def make_momentum(eta=0.009, alpha=0.9):
    dw = np.zeros(2)
    def update(w, g):
        nonlocal dw
        dw = -eta * g + alpha * dw
        return w + dw
    return update

# Adam: divides each weight's step by the size of its recent gradients,
# so the steep w2 gets a SMALLER effective step and the flat w1 a LARGER one
def make_adam(eta=0.3, r1=0.9, r2=0.999, eps=1e-8):
    m, v, t = np.zeros(2), np.zeros(2), 0
    def update(w, g):
        nonlocal m, v, t
        t += 1
        m = r1*m + (1-r1)*g
        v = r2*v + (1-r2)*g**2
        mh = m / (1 - r1**t)
        vh = v / (1 - r2**t)
        return w - eta * mh / (np.sqrt(vh) + eps)
    return update

print(f"{'Optimizer':<22} {'Step 10':>10} {'Step 50':>10} {'Step 100':>11}  {'Final w'}")
print("-" * 75)
for name, updater in [('SGD (eta=0.021)',  make_sgd(0.021)),
                      ('SGD (eta=0.009)',  make_sgd()),
                      ('Momentum (a=0.9)', make_momentum()),
                      ('Adam (eta=0.3)',   make_adam())]:
    w = np.array([5.0, 5.0])
    losses = [loss(w)]
    for _ in range(100):
        w = updater(w, grad(w))
        losses.append(loss(w))
    print(f"  {name:<20} {losses[10]:>10.4g} {losses[50]:>10.4g} {losses[100]:>11.4g}  "
          f"[{w[0]:+.4f}, {w[1]:+.4f}]")

print()
print("Observations:")
print("  SGD      : One step size for both weights. Too big and the steep w2 blows up;")
print("             small enough for w2, it crawls along the flat w1 direction")
print("  Momentum : Builds up speed along w1 and finishes far ahead of SGD")
print("  Adam     : Per-weight scaling moves both weights at a similar pace; it beats SGD")
print("             here, but a well-tuned momentum can still win on a smooth toy bowl")

=== Optimizer Comparison: Minimizing E(w) = w1^2 + 50*w2^2 ===

Starting at w=[5.0, 5.0], optimal = [0.0, 0.0]
A narrow ravine: steep in w2 (gradient 100*w2), almost flat in w1 (gradient 2*w1)

Optimizer                 Step 10    Step 50    Step 100  Final w
---------------------------------------------------------------------------
  SGD (eta=0.021)            8420  1.723e+07   2.374e+11  [+0.0685, +68903.0617]
  SGD (eta=0.009)           17.38      4.065      0.6611  [+0.8131, +0.0000]
  Momentum (a=0.9)          77.41      2.635   0.0009467  [+0.0269, +0.0021]
  Adam (eta=0.3)            224.2     0.6329     0.02368  [+0.0215, +0.0215]

Observations:
  SGD      : One step size for both weights. Too big and the steep w2 blows up;
             small enough for w2, it crawls along the flat w1 direction
  Momentum : Builds up speed along w1 and finishes far ahead of SGD
  Adam     : Per-weight scaling moves both weights at a similar pace; it beats SGD
             here, but a well-tune

## Example 2: L1 vs L2 Regularization — Effect on Weights

We train a logistic regression model on a 20-feature dataset where only 5 features are truly informative. Without regularization, all 20 weights are non-zero — the model uses noise features. L2 regularization shrinks all weights toward zero proportionally. L1 regularization does something smarter: it pushes weights to exactly zero (sparse solution). With strong L1 (C=0.1), 15 features get zeroed out — the model correctly identifies the informative features! This is the key practical difference: L2 shrinks, L1 selects.

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import numpy as np

np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=20, n_informative=5,
                            n_redundant=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr); X_te = scaler.transform(X_te)

print("=== L1 vs L2 Regularization ===\n")
print("Dataset: 20 features, only 5 truly informative\n")
print(f"{'Regularization':<22} {'Test Acc':>9} {'||w||':>8} {'Zero weights':>14}")
print("-" * 58)
results = {}
for name, C, l1r in [
    ('No Reg (C=1e6)',  1e6, 0.0),
    ('L2 (C=1.0)',      1.0, 0.0),
    ('L2 strong (C=0.1)', 0.1, 0.0),
    ('L1 (C=1.0)',      1.0, 1.0),
    ('L1 strong (C=0.1)', 0.1, 1.0),
]:
    clf = LogisticRegression(C=C, l1_ratio=l1r, solver='saga',
                             max_iter=2000, random_state=42)
    clf.fit(X_tr, y_tr)
    acc  = accuracy_score(y_te, clf.predict(X_te))
    w    = clf.coef_[0]
    n_z  = np.sum(np.abs(w) < 1e-4)
    wn   = np.linalg.norm(w)
    results[name] = w
    print(f"  {name:<20} {acc*100:>8.1f}% {wn:>8.3f} {n_z:>14}")

print("\n=== No-Reg vs L1-Strong: Per-Feature Weight Comparison ===\n")
w_none = results['No Reg (C=1e6)']
w_l1   = results['L1 strong (C=0.1)']
print(f"  {'Feature':>10} {'No Reg':>10} {'L1 Strong':>12}  Status")
print("-" * 48)
for i, (wn, wl) in enumerate(zip(w_none, w_l1)):
    status = "ZEROED OUT" if abs(wl) < 1e-4 else "Active"
    print(f"  Feature {i+1:>2}  {wn:>10.4f} {wl:>12.4f}  {status}")

=== L1 vs L2 Regularization ===

Dataset: 20 features, only 5 truly informative

Regularization          Test Acc    ||w||   Zero weights
----------------------------------------------------------
  No Reg (C=1e6)           84.0%    1.883              0
  L2 (C=1.0)               84.7%    1.805              0
  L2 strong (C=0.1)        85.3%    1.404              0
  L1 (C=1.0)               84.7%    2.278             11
  L1 strong (C=0.1)        84.0%    1.432             15

=== No-Reg vs L1-Strong: Per-Feature Weight Comparison ===

     Feature     No Reg    L1 Strong  Status
------------------------------------------------
  Feature  1      0.2175       0.0000  ZEROED OUT
  Feature  2      0.3026       0.0000  ZEROED OUT
  Feature  3      0.6299       0.0000  ZEROED OUT
  Feature  4      0.4659       0.6188  Active
  Feature  5      0.0210       0.0000  ZEROED OUT
  Feature  6      0.1780       0.0000  ZEROED OUT
  Feature  7      0.0852       0.0000  ZEROED OUT
  Feature  8     

## Example 3: AdaGrad vs RMSProp — Learning Rate Decay Comparison

This example simulates 30 steps of gradient descent and traces the effective learning rate of AdaGrad vs RMSProp for a single weight. The first 10 steps have large gradients (simulating early training); the last 20 have small gradients (simulating late training near convergence). AdaGrad accumulates ALL squared gradients — so by step 11, its effective learning rate has already decayed to 0.00389 and keeps shrinking even as gradients become small. RMSProp only keeps a moving average (ρ=0.9), so when gradients become small, its denominator shrinks too, keeping the learning rate healthy. The Ratio column shows RMSProp learning 11x faster than AdaGrad by step 30.

In [3]:
import numpy as np

np.random.seed(0)
T   = 30
eta = 0.1
rho = 0.9
eps = 1e-8

# Simulate gradients: large in first 10 steps, small afterward
grads = np.concatenate([np.random.uniform(5, 10, 10),
                         np.random.uniform(0.1, 0.5, 20)])

G_ada = 0.0
v_rms = 0.0

print("=== AdaGrad vs RMSProp: Effective Learning Rate Trace ===\n")
print("Steps 1-10: large gradients | Steps 11-30: small gradients\n")
print(f"{'Step':>5} {'Gradient':>10} {'AdaGrad LR':>12} {'RMSProp LR':>12}  {'Ratio'}")
print("-" * 55)
for t in range(1, T + 1):
    g = grads[t - 1]
    G_ada += g**2
    lr_ada = eta / (np.sqrt(G_ada) + eps)
    v_rms  = rho * v_rms + (1 - rho) * g**2
    lr_rms = eta / (np.sqrt(v_rms) + eps)
    ratio  = lr_rms / lr_ada
    print(f"{t:>5} {g:>10.4f} {lr_ada:>12.6f} {lr_rms:>12.6f}  {ratio:>5.1f}x")

print()
print("Key insight:")
print("  AdaGrad LR decays to ~0.0039 and stays there even for tiny gradients.")
print("  RMSProp LR recovers because it forgets old (large) gradients over time.")
print("  By step 30, RMSProp learns 11x faster than AdaGrad on the same data!")

=== AdaGrad vs RMSProp: Effective Learning Rate Trace ===

Steps 1-10: large gradients | Steps 11-30: small gradients

 Step   Gradient   AdaGrad LR   RMSProp LR  Ratio
-------------------------------------------------------
    1     7.7441     0.012913     0.040835    3.2x
    2     8.5759     0.008654     0.028003    3.2x
    3     8.0138     0.007111     0.023637    3.3x
    4     7.7244     0.006233     0.021283    3.4x
    5     7.1183     0.005697     0.020026    3.5x
    6     8.2295     0.005158     0.018501    3.6x
    7     7.1879     0.004837     0.017829    3.7x
    8     9.4589     0.004398     0.016382    3.7x
    9     9.8183     0.004038     0.015219    3.8x
   10     6.9172     0.003889     0.015137    3.9x
   11     0.4167     0.003889     0.015953    4.1x
   12     0.3116     0.003888     0.016813    4.3x
   13     0.3272     0.003888     0.017720    4.6x
   14     0.4702     0.003887     0.018671    4.8x
   15     0.1284     0.003887     0.019680    5.1x
   16     